## 1. 라이브러리 설치 및 환경 설정

In [13]:
#!pip install lightgbm catboost optuna xgboost -q

In [14]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import optuna
import warnings
import matplotlib.pyplot as plt

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import mean_absolute_error
from scipy import stats

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED    = 42
N_FOLDS = 5
TARGET  = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

np.random.seed(SEED)
print('✅ 환경 설정 완료')

✅ 환경 설정 완료


## 2. 데이터 로드 및 레이아웃 병합

In [15]:
train  = pd.read_csv('./data/train.csv')
test   = pd.read_csv('./data/test.csv')
layout = pd.read_csv('./data/layout_info.csv')

layout_type_map = {v: i for i, v in enumerate(layout['layout_type'].unique())}
layout['layout_type_enc'] = layout['layout_type'].map(layout_type_map)

train = train.merge(layout.drop(columns=['layout_type']), on='layout_id', how='left')
test  = test.merge(layout.drop(columns=['layout_type']),  on='layout_id', how='left')

print(f'학습: {train.shape}, 테스트: {test.shape}')
print(f'layout_type 인코딩: {layout_type_map}')

학습: (250000, 108), 테스트: (50000, 107)
layout_type 인코딩: {'narrow': 0, 'grid': 1, 'hybrid': 2, 'hub_spoke': 3}


## 3. 타겟 분석 및 이상치 처리

In [ ]:
print('=== 타겟 변수 기본 통계 ===')
print(train[TARGET].describe())
skewness = stats.skew(train[TARGET].dropna())
print(f'\n왜도: {skewness:.4f}')
for q in [0.99, 0.995, 0.999]:
    thresh = train[TARGET].quantile(q)
    n_clip = (train[TARGET] > thresh).sum()
    print(f'  상위 {(1-q)*100:.1f}%: {thresh:.1f}분 이상 {n_clip}건')

=== 타겟 변수 기본 통계 ===
count    250000.000000
mean         18.962296
std          27.351374
min           0.000000
25%           4.278801
50%           9.032652
75%          25.791869
max         715.858119
Name: avg_delay_minutes_next_30m, dtype: float64

왜도: 5.6820
  상위 0.1% 클리핑: 301.4분 이상 250건 (0.10%)


In [ ]:
print('=== 타겟 변수 기본 통계 ===')
print(train[TARGET].describe())

# 🚀 [핵심 추가] 눈으로만 보지 말고, 실제로 상위 1% 괴물들을 싹둑 잘라냅니다!
CLIP_THRESH = train[TARGET].quantile(0.99)


print(f'\n🚨 이상치 제거 완료! (모든 지연 시간의 상한선을 {CLIP_THRESH:.1f}분으로 강제 고정)')

=== 타겟 변수 기본 통계 ===
count    250000.000000
mean         18.962296
std          27.351374
min           0.000000
25%           4.278801
50%           9.032652
75%          25.791869
max         715.858119
Name: avg_delay_minutes_next_30m, dtype: float64

🚨 이상치 제거 완료! (모든 지연 시간의 상한선을 120.9분으로 강제 고정)


## 4. 도메인 기반 피처 엔지니어링

In [18]:
# 4. 도메인 기반 피처 엔지니어링 (시계열 누적 로직 완전 삭제 & 순수 스냅샷 지표 복구)
def engineer_features(df):
    d = df.copy()
    
    # 🚨 주의: d.sort_values(...) 정렬 코드 삭제! (독립 스냅샷이므로 정렬하면 안 됨)

    # ── ① 로봇 압박 지표 ──────────────────────────────────
    d['charge_pressure']       = d['charge_queue_length'] / (d['charger_count'] + 1)
    d['active_robot_ratio']    = d['robot_active'] / (d['robot_total'] + 1)
    d['idle_robot_ratio']      = d['robot_idle'] / (d['robot_total'] + 1)
    d['low_batt_robot_count']  = d['low_battery_ratio'] * d['robot_active']
    d['battery_stress']        = (1 - d['battery_mean'] / 100) * d['battery_std']
    d['charge_inefficiency']   = (100 - d['charge_efficiency_pct']) / 100
    d['effective_robot_avail'] = d['robot_total'] - d['low_batt_robot_count'] - d['charge_queue_length']
    d['robot_fault_ratio']     = d['fault_count_15m'] / (d['robot_active'] + 1)

    # ── ② 병목 복합 지수 ──────────────────────────────────
    d['incident_total_15m']    = d['blocked_path_15m'] + d['near_collision_15m'] + d['fault_count_15m']
    d['congestion_load']       = d['congestion_score'] * d['avg_trip_distance']
    d['wait_per_intersection'] = d['intersection_wait_time_avg'] / (d['intersection_count'] + 1)
    d['path_congestion_gap']   = d['path_optimization_score'] - d['congestion_score']
    d['aisle_density']         = d['aisle_traffic_score'] / (d['aisle_width_avg'] + 0.1)
    d['oneway_congestion']     = d['one_way_ratio'] * d['congestion_score']
    d['compact_congestion']    = d['layout_compactness'] * d['congestion_score']
    d['max_density_sq']        = d['max_zone_density'] ** 2 

    # ── ③ 주문 부하 지수 ──────────────────────────────────
    d['order_complexity']      = d['unique_sku_15m'] * d['avg_items_per_order']
    d['urgent_heavy_ratio']    = d['urgent_order_ratio'] * d['heavy_item_ratio']
    d['effective_order_load']  = d['order_inflow_15m'] * (1 + d['urgent_order_ratio'])
    d['rework_pressure']       = d['return_order_ratio'] + d['replenishment_overlap']
    d['forecast_miss']         = 1 - d['daily_forecast_accuracy']
    d['pick_complexity']       = d['pick_list_length_avg'] * (1 - d['sku_concentration'])
    d['backorder_urgency']     = d['backorder_ratio'] * d['urgent_order_ratio']

    # ── ④ 설비 용량 대비 수요 압박 ────────────────────────
    d['dock_pack_util_avg']      = (d['pack_utilization'] + d['loading_dock_util'] + d['staging_area_util']) / 3
    d['orders_per_pack_station'] = d['order_inflow_15m'] / (d['pack_station_count'] + 1)
    d['orders_per_robot']        = d['order_inflow_15m'] / (d['robot_active'] + 1)
    d['robot_density']           = d['robot_total'] / (d['floor_area_sqm'] + 1)
    d['charger_robot_ratio']     = d['charger_count'] / (d['robot_total'] + 1)
    d['pack_area_pressure']      = d['pack_utilization'] * d['layout_compactness']
    d['dock_truck_bottleneck']   = d['loading_dock_util'] * d['outbound_truck_wait_min']

    # ── ⑤ 환경·시스템 스트레스 ──────────────────────────
    d['temp_diff']             = abs(d['warehouse_temp_avg'] - d['external_temp_c'])
    d['heat_humidity_index']   = d['warehouse_temp_avg'] * d['humidity_pct'] / 100
    d['it_bottleneck']         = d['wms_response_time_ms'] * (1 + d['scanner_error_rate'])
    d['network_instability']   = d['network_latency_ms'] * (1 - d['wifi_signal_db'] / 100)
    d['barcode_fail_rate']     = 1 - d['barcode_read_success_rate']
    d['label_scan_bottleneck'] = d['label_print_queue'] * (1 + d['scanner_error_rate'])
    d['conveyor_load']         = d['avg_package_weight_kg'] / (d['conveyor_speed_mps'] + 0.01)

    # ── ⑥ 인력·운영 효율 ────────────────────────────────
    d['orders_per_staff']      = d['order_inflow_15m'] / (d['staff_on_floor'] + 1)
    d['forklift_staff_ratio']  = d['forklift_active_count'] / (d['staff_on_floor'] + 1)
    d['shift_load_ratio']      = d['order_inflow_15m'] / (d['prev_shift_volume'] + 1)
    d['handover_pressure']     = d['shift_handover_delay_min'] * d['prev_shift_volume'] / 1000
    d['skilled_capacity']      = d['staff_on_floor'] * d['worker_avg_tenure_months']

    # ── ⑦ 시간대 패턴 ────────────────────────────────────
    d['is_peak_hour']          = d['shift_hour'].isin([9,10,11,14,15,16,17]).astype(int)
    d['is_shift_start']        = d['shift_hour'].isin([0,1,8]).astype(int)
    d['is_weekend']            = d['day_of_week'].isin([6,7]).astype(int)
    d['is_monday']             = (d['day_of_week'] == 1).astype(int)
    d['peak_order_interaction']= d['is_peak_hour'] * d['order_inflow_15m']
    d['shift_hour_sin']        = np.sin(2 * np.pi * d['shift_hour'] / 24)
    d['shift_hour_cos']        = np.cos(2 * np.pi * d['shift_hour'] / 24)
    d['dow_sin']               = np.sin(2 * np.pi * d['day_of_week'] / 7)
    d['dow_cos']               = np.cos(2 * np.pi * d['day_of_week'] / 7)

    # ── ⑧ 레이아웃 구조 ──────────────────────────────────
    d['age_maintenance_risk']   = d['building_age_years'] * (1 - d['maintenance_schedule_score'])
    d['vertical_gap']           = d['ceiling_height_m'] - d['racking_height_avg_m']
    d['dispersion_trip_cost']   = d['zone_dispersion'] * d['avg_trip_distance']
    d['crossdock_dock_pressure']= d['cross_dock_ratio'] * d['loading_dock_util']
    d['cold_chain_risk']        = d['cold_chain_ratio'] * abs(d['cold_storage_temp_c'] + 18)
    d['area_per_robot']         = d['floor_area_sqm'] / (d['robot_total'] + 1)

    # ── ⑨ 3방향 교호작용 (회원님의 핵심 로직 유지) ────────────────
    d['robot_order_congestion'] = d['orders_per_robot'] * d['congestion_score']
    d['battery_order_pressure'] = d['battery_stress'] * d['effective_order_load']
    d['it_order_bottleneck']    = d['it_bottleneck'] * d['order_inflow_15m'] / 1000
    d['super_traffic_jam_risk'] = d['sku_concentration'] * d['avg_trip_distance'] * d['layout_compactness']

    # 🚨 이 아래에 있던 time_step, cum_order_inflow, rolling, diff 관련 코드는 100% 삭제했습니다!
    return d

train_fe = engineer_features(train)
test_fe  = engineer_features(test)

feature_cols = [c for c in train_fe.columns if c not in ID_COLS + [TARGET]]
print(f'전체 피처 수: {len(feature_cols)}')

전체 피처 수: 165


## 5. CV 기반 타겟 인코딩 (Leakage 방지)

In [19]:
# def target_encode_cv(train_df, test_df, col, target, n_splits=5, alpha=20):
#     """
#     CV 기반 타겟 인코딩
#     - train: fold 밖 데이터로만 인코딩 (leakage 방지)
#     - test : 전체 train 평균으로 인코딩
#     - alpha: 스무딩 파라미터 (그룹 샘플 수 적을 때 전체 평균으로 회귀)
#     """
#     global_mean = train_df[target].mean()
#     train_enc   = np.zeros(len(train_df))
#     kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

#     for tr_i, val_i in kf.split(train_df):
#         stats_map = (
#             train_df.iloc[tr_i]
#             .groupby(col)[target]
#             .agg(['mean', 'count'])
#         )
#         # 스무딩: 샘플 수 적은 그룹은 전체 평균으로 당김
#         stats_map['smooth'] = (
#             (stats_map['count'] * stats_map['mean'] + alpha * global_mean)
#             / (stats_map['count'] + alpha)
#         )
#         train_enc[val_i] = train_df.iloc[val_i][col].map(stats_map['smooth']).fillna(global_mean)

#     # 테스트는 전체 train으로 인코딩
#     full_stats = (
#         train_df.groupby(col)[target]
#         .agg(['mean', 'count'])
#     )
#     full_stats['smooth'] = (
#         (full_stats['count'] * full_stats['mean'] + alpha * global_mean)
#         / (full_stats['count'] + alpha)
#     )
#     test_enc = test_df[col].map(full_stats['smooth']).fillna(global_mean).values

#     return train_enc, test_enc


# # layout_id 타겟 인코딩
# te_layout_tr, te_layout_te = target_encode_cv(train_fe, test_fe, 'layout_id', TARGET)
# train_fe['te_layout_id'] = te_layout_tr
# test_fe['te_layout_id']  = te_layout_te

# # layout_type_enc 타겟 인코딩
# te_ltype_tr, te_ltype_te = target_encode_cv(
#     train_fe, test_fe, 'layout_type_enc', TARGET
# )
# train_fe['te_layout_type'] = te_ltype_tr
# test_fe['te_layout_type']  = te_ltype_te

# # 피처 목록 갱신
# feature_cols = [c for c in train_fe.columns if c not in ID_COLS + [TARGET]]
# print(f'타겟 인코딩 추가 후 피처 수: {len(feature_cols)}')

## 6. 결측치 처리 및 모델 학습 준비

In [20]:
# 결측치 처리
for col in feature_cols:
    med = train_fe[col].median()
    train_fe[col] = train_fe[col].fillna(med)
    test_fe[col]  = test_fe[col].fillna(med)

X      = train_fe[feature_cols].copy()
y      = train_fe[TARGET].copy()
X_test = test_fe[feature_cols].copy()

# ✅ log 변환 복구
y_tr_all = np.log1p(y.clip(lower=0))

def decode_pred(pred):
    return np.expm1(pred).clip(0)

gkf    = GroupKFold(n_splits=N_FOLDS)
groups = train_fe['scenario_id']

## 7. 저중요도 피처 제거

In [21]:
# 7. 빠른 LightGBM으로 피처 중요도 확인 후 하위 20% 제거
from lightgbm import LGBMRegressor

quick_model = LGBMRegressor(
    n_estimators=300, learning_rate=0.1, max_depth=6,
    num_leaves=63, random_state=SEED, verbose=-1
)

# 🚀 kf.split -> gkf.split(X, y, groups=groups) 로 변경! (누수 차단)
tr_i, val_i = next(gkf.split(X, y, groups=groups))

# 🚀 y_tr_all -> 순정 타겟 y 로 변경! (에러 방지)
quick_model.fit(X.iloc[tr_i], y.iloc[tr_i])

imp_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': quick_model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

# 중요도 하위 20% 제거
threshold = imp_df['importance'].quantile(0.05)
keep_features = imp_df[imp_df['importance'] > threshold]['feature'].tolist()
drop_features = imp_df[imp_df['importance'] <= threshold]['feature'].tolist()

print(f'유지 피처: {len(keep_features)}개')
print(f'제거 피처: {len(drop_features)}개 (중요도 ≤ {threshold:.1f})')
print(f'제거된 피처 예시: {drop_features[:10]}')

X      = X[keep_features]
X_test = X_test[keep_features]
feature_cols = keep_features
print(f'\n최종 피처 수: {len(feature_cols)}')

유지 피처: 156개
제거 피처: 9개 (중요도 ≤ 7.4)
제거된 피처 예시: ['dow_sin', 'incident_total_15m', 'dow_cos', 'task_reassign_15m', 'is_monday', 'max_density_sq', 'is_weekend', 'is_peak_hour', 'is_shift_start']

최종 피처 수: 156


## 8. Optuna 하이퍼파라미터 튜닝 (LightGBM)

In [ ]:
# 8. Optuna 하이퍼파라미터 튜닝 (LightGBM)

# 튜닝용 샘플링 (50K)
tune_idx = np.random.choice(len(X), 50000, replace=False)
X_tune   = X.iloc[tune_idx].reset_index(drop=True)
# 🚀 y_tr_all -> y 로 변경! (에러 방지)
y_tune   = y_tr_all.iloc[tune_idx].reset_index(drop=True)

# 🚀 튜닝할 때도 그룹 누수가 없도록 groups도 같이 샘플링해 줍니다.
groups_tune = groups.iloc[tune_idx].reset_index(drop=True)

def lgb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators',      500, 3000),
        learning_rate     = trial.suggest_float('learning_rate',   0.01, 0.1, log=True),
        max_depth         = trial.suggest_int('max_depth',         4, 8),
        num_leaves        = trial.suggest_int('num_leaves',        31, 63),
        subsample         = trial.suggest_float('subsample',       0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree',0.5, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha',       1e-4, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda',      1e-4, 10.0, log=True),
        min_child_samples = trial.suggest_int('min_child_samples', 50, 150),
    )
    FIXED = dict(device='gpu', random_state=SEED, verbose=-1)
    
    # 🚀 KFold -> GroupKFold 로 변경! (튜닝 시 과적합 방지)
    gkf_tune = GroupKFold(n_splits=3)
    maes = []
    
    # 🚀 split에 groups=groups_tune 을 꼭 넣어줍니다.
    for tr_i, val_i in gkf_tune.split(X_tune, y_tune, groups=groups_tune):
        model = LGBMRegressor(**params, **FIXED)
        model.fit(
            X_tune.iloc[tr_i], y_tune.iloc[tr_i],
            eval_set=[(X_tune.iloc[val_i], y_tune.iloc[val_i])],
            callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)]
        )
        # 🚀 decode_pred 제거, 원본 타겟 그대로 예측
        pred = decode_pred(model.predict(X_tune.iloc[val_i]))
        true = decode_pred(y_tune.iloc[val_i].values)
        maes.append(mean_absolute_error(true, pred))
    return np.mean(maes)

print('🔍 LightGBM Optuna 튜닝 (50 trials)...')
study_lgb = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study_lgb.optimize(lgb_objective, n_trials=50, show_progress_bar=True)

BEST_LGB = study_lgb.best_params
print(f'\n✅ LightGBM 최적 MAE: {study_lgb.best_value:.4f}')
print('   파라미터:', BEST_LGB)

🔍 LightGBM Optuna 튜닝 (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]

[W 2026-04-09 13:06:09,848] Trial 0 failed with parameters: {'n_estimators': 1436, 'learning_rate': 0.08927180304353628, 'max_depth': 7, 'num_leaves': 50, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'reg_alpha': 0.00019517224641449495, 'reg_lambda': 2.1423021757741068, 'min_child_samples': 110} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\YJ\AppData\Roaming\Python\Python313\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11000\1705167147.py", line 33, in lgb_objective
    model.fit(
    ~~~~~~~~~^
        X_tune.iloc[tr_i], y_tune.iloc[tr_i],
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        eval_set=[(X_tune.iloc[val_i], y_tune.iloc[val_i])],
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-

KeyboardInterrupt: 

## 9. LightGBM 5-Fold 학습

In [ ]:
# 9. LightGBM 5-Fold 학습 (학습 곡선 에러 완벽 수정본)
oof_lgb        = np.zeros(len(X))
test_lgb       = np.zeros(len(X_test))
lgb_importances= np.zeros(len(feature_cols))
fold_maes_lgb  = []
FIXED = dict(device='gpu', random_state=SEED, verbose=-1)

print('🚀 LightGBM Group 5-Fold 학습 (학습 곡선 모니터링 포함)\n')

# 🚀 학습 과정의 오차를 기록할 빈 바구니 준비
evals_result = {} 

for fold, (tr_i, val_i) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'── Fold {fold+1}/{N_FOLDS} ──')
    X_tr, y_tr   = X.iloc[tr_i], y_tr_all.iloc[tr_i]
    X_val, y_val = X.iloc[val_i], y_tr_all.iloc[val_i]

    model = LGBMRegressor(**BEST_LGB, **FIXED)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_tr, y_tr), (X_val, y_val)], 
        eval_names=['train', 'valid'],
        eval_metric='mae', # 🚀 [핵심 1] LightGBM에게 "반드시 MAE 오차를 기록해라!" 라고 명시
        callbacks=[
            lgb.early_stopping(50, verbose=False), 
            lgb.log_evaluation(200),
            lgb.record_evaluation(evals_result)
        ]
    )
    
    # 🚀 Fold 1 학습이 끝난 직후 그래프 그리기!
    if fold == 0:
        plt.figure(figsize=(10, 6))
        
        # 🚀 [핵심 2] 'l1'인지 'mae'인지 'l2'인지 모델이 저장한 진짜 이름을 찾아옵니다.
        actual_metric = list(evals_result['valid'].keys())[0]
        
        lgb.plot_metric(evals_result, metric=actual_metric) 
        plt.title(f'Fold 1: Learning Curve (Train vs Validation - {actual_metric})')
        plt.tight_layout()
        plt.show()

    # 원본 코드 유지 (정답 예측 및 OOF 병합)
    oof_lgb[val_i]   = decode_pred(model.predict(X_val))
    test_lgb        += decode_pred(model.predict(X_test)) / N_FOLDS
    lgb_importances += model.feature_importances_ / N_FOLDS

    mae = mean_absolute_error(decode_pred(y_val), oof_lgb[val_i])
    fold_maes_lgb.append(mae)
    print(f'   Fold MAE: {mae:.4f}\n')

lgb_oof_mae = mean_absolute_error(y, oof_lgb)
print(f'📊 LightGBM OOF MAE: {lgb_oof_mae:.4f}')

## 10. CatBoost 5-Fold 학습

In [ ]:
# 10. CatBoost 5-Fold 학습
oof_cb       = np.zeros(len(X))
test_cb      = np.zeros(len(X_test))
fold_maes_cb = []

print('🐱 CatBoost 5-Fold 학습\n')
# 🚀 kf.split -> gkf.split 적용
for fold, (tr_i, val_i) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'── Fold {fold+1}/{N_FOLDS} ──')
    X_tr, y_tr   = X.iloc[tr_i], y_tr_all.iloc[tr_i]  # ✅ log 변환 타겟
    X_val, y_val = X.iloc[val_i], y_tr_all.iloc[val_i] # ✅ log 변환 타겟

    model = CatBoostRegressor(
        iterations=3000, learning_rate=0.03, depth=8,
        l2_leaf_reg=5.0,
        bootstrap_type='Bernoulli', subsample=0.8,
        random_seed=SEED, eval_metric='MAE',
        early_stopping_rounds=50, verbose=200,
        task_type='GPU', devices='0',
    )
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    # 🚀 decode_pred 제거, 원본 그대로 예측
    oof_cb[val_i]  = decode_pred(model.predict(X_val))
    test_cb       += decode_pred(model.predict(X_test)) / N_FOLDS

    mae = mean_absolute_error(decode_pred(y_val), oof_cb[val_i])
    fold_maes_cb.append(mae)
    print(f'   Fold MAE: {mae:.4f}\n')

cb_oof_mae = mean_absolute_error(y, oof_cb)
print(f'📊 CatBoost OOF MAE: {cb_oof_mae:.4f}')

## 11. XGBoost 5-Fold 학습

In [ ]:
# 11. XGBoost 5-Fold 학습
oof_xgb       = np.zeros(len(X))
test_xgb      = np.zeros(len(X_test))
fold_maes_xgb = []

print('⚡ XGBoost 5-Fold 학습\n')
# 🚀 kf.split -> gkf.split 적용
for fold, (tr_i, val_i) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'── Fold {fold+1}/{N_FOLDS} ──')
    # 🚀 y_tr_all 대신 원본 y 사용
    X_tr, y_tr   = X.iloc[tr_i], y_tr_all.iloc[tr_i]
    X_val, y_val = X.iloc[val_i], y_tr_all.iloc[val_i]

    model = XGBRegressor(
        n_estimators=3000, learning_rate=0.03,
        max_depth=7, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        tree_method='hist',  
        device='cuda',
        random_state=SEED, verbosity=0,
        early_stopping_rounds=100,
        eval_metric='mae',
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=200
    )

    # 🚀 decode_pred 제거, 원본 그대로 예측
    oof_xgb[val_i]  = decode_pred(model.predict(X_val))
    test_xgb       += decode_pred(model.predict(X_test)) / N_FOLDS

    mae = mean_absolute_error(decode_pred(y_val), oof_xgb[val_i])
    fold_maes_xgb.append(mae)
    print(f'   Fold MAE: {mae:.4f}\n')

xgb_oof_mae = mean_absolute_error(y, oof_xgb)
print(f'📊 XGBoost OOF MAE: {xgb_oof_mae:.4f}')

## 12. 3모델 앙상블 – 최적 가중치 탐색

In [ ]:
# 12. 3모델 앙상블 – 최적 가중치 탐색
from scipy.optimize import minimize

def ensemble_mae(weights):
    w1, w2, w3 = weights
    blend = w1 * oof_lgb + w2 * oof_cb + w3 * oof_xgb
    # 🚀 y_orig 대신 우리가 정의한 순정 타겟 y를 사용합니다!
    return mean_absolute_error(y, blend)

# 제약: 가중치 합 = 1, 각 가중치 >= 0
constraints = {'type': 'eq', 'fun': lambda w: w[0] + w[1] + w[2] - 1}
bounds = [(0, 1), (0, 1), (0, 1)]
init   = [1/3, 1/3, 1/3]

result = minimize(ensemble_mae, init, method='SLSQP',
                  bounds=bounds, constraints=constraints)

w_lgb, w_cb, w_xgb = result.x
best_ensemble_mae   = result.fun

print(f'⚖️  최적 앙상블 가중치')
print(f'   LightGBM : {w_lgb:.3f}')
print(f'   CatBoost : {w_cb:.3f}')
print(f'   XGBoost  : {w_xgb:.3f}')

print(f'\n{"="*55}')
print(f'  🏆 최종 성능 요약')
print(f'{"="*55}')
print(f'  베이스라인          : 9.2405')
print(f'  LightGBM           : {lgb_oof_mae:.4f}  ({9.2405-lgb_oof_mae:+.4f})')
print(f'  CatBoost           : {cb_oof_mae:.4f}  ({9.2405-cb_oof_mae:+.4f})')
print(f'  XGBoost            : {xgb_oof_mae:.4f}  ({9.2405-xgb_oof_mae:+.4f})')
print(f'  3모델 앙상블        : {best_ensemble_mae:.4f}  ({9.2405-best_ensemble_mae:+.4f})')
print(f'{"="*55}')

## 13. 피처 중요도 분석

In [ ]:
imp_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgb_importances
}).sort_values('importance', ascending=False).reset_index(drop=True)

top30 = imp_df.head(30)

fig, ax = plt.subplots(figsize=(10, 12))
ax.barh(top30['feature'][::-1], top30['importance'][::-1], color='steelblue')
ax.set_xlabel('Feature Importance (gain)')
ax.set_title('Top 30 Feature Importances (LightGBM)', fontsize=12)
plt.tight_layout()
plt.savefig('./feature_importance_v3.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 15 피처:')
print(imp_df.head(15)[['feature','importance']].to_string(index=False))

## 14. 제출 파일 생성

In [ ]:
# 14. 제출 파일 생성 (ID 셔플 완벽 차단)
final_preds = (w_lgb * test_lgb + w_cb * test_cb + w_xgb * test_xgb).clip(0)

# 🚀 1. 정렬된 예측값에 올바른 ID 이름표를 달아줍니다. (test_fe는 위에서 정렬된 상태임)
pred_df = pd.DataFrame({'ID': test_fe['ID'], TARGET: final_preds})

# 🚀 2. 주최측이 요구하는 원본 제출 폼을 불러옵니다.
submission = pd.read_csv('./sample_submission.csv')
submission = submission.drop(columns=[TARGET], errors='ignore')

# 🚀 3. ID를 기준으로 왼쪽(원본)에 오른쪽(예측값)을 정확히 끼워 맞춥니다.
submission = pd.merge(submission, pred_df, on='ID', how='left')

submission.to_csv('./submission_v5_MASTER.csv', index=False)

print('✅ submission_v5_MASTER.csv 저장 완료')
print(f'\n🚨 [최종 점검] 예측값 평균 확인 (15.xx ~ 16.xx 분이 나오면 대성공!):')
print(submission[TARGET].describe())